In [0]:
import json

# 1. Lecture de la clé JSON
json_path = "/Workspace/Users/matixeu@gmail.com/gcp-key.json"

with open(json_path, "r") as f:
    key_data = json.load(f)

client_email = key_data["client_email"]
private_key = key_data["private_key"]
private_key_id = key_data["private_key_id"]

# 2. Lecture du fichier sur GCS avec wildcard '*'
bucket_name = "etf-raw-data-alban-2026"
file_path = f"gs://{bucket_name}/raw/NVDA_data_*.csv"

df_raw = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .option("fs.gs.auth.service.account.enable", "true") \
    .option("fs.gs.auth.service.account.email", client_email) \
    .option("fs.gs.auth.service.account.private.key", private_key) \
    .option("fs.gs.auth.service.account.private.key.id", private_key_id) \
    .load(file_path)

print(" Fichier chargé avec succès depuis GCS !")
print("=== 1. SCHÉMA BRUT (BRONZE) ===")
df_raw.printSchema()

In [0]:
from pyspark.sql.functions import col, to_date, round, current_timestamp

# 1. Cast / Typage explicite des colonnes
df_cleaned = df_raw \
    .withColumn("Date", to_date(col("Date"), "yyyy-MM-dd")) \
    .withColumn("Open", col("Open").cast("double")) \
    .withColumn("High", col("High").cast("double")) \
    .withColumn("Low", col("Low").cast("double")) \
    .withColumn("Close", col("Close").cast("double")) \
    .withColumn("Volume", col("Volume").cast("long"))

# 2. Suppression des doublons sur la date
df_dedup = df_cleaned.dropDuplicates(["Date"])

# 3. Enrichissement (Calculs de variation & Horodatage)
df_silver = df_dedup \
    .withColumn("Daily_Variation", round(col("Close") - col("Open"), 4)) \
    .withColumn("Daily_Variation_Pct", round(((col("Close") - col("Open")) / col("Open")) * 100, 2)) \
    .withColumn("processed_at", current_timestamp()) \
    .orderBy(col("Date").desc())

# 4. Validation du résultat (Schéma + Tableau)
print("=== 2. SCHÉMA NETTOYÉ (SILVER) ===")
df_silver.printSchema()

display(df_silver)